In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, f1_score, roc_auc_score

In [2]:
train = pd.read_csv("../data/processed/train.csv")
cols = list(train.columns)
URL_COLS = cols[cols.index("length_url") : cols.index("statistical_report") + 1]
CONTENT_COLS = cols[cols.index("nb_hyperlinks") : cols.index("domain_with_copyright") + 1]

# Modèle retenu : RF sur URL + Contenu, sans random_domain (non reproductible en live)
FINAL_COLS = [c for c in URL_COLS + CONTENT_COLS if c != "random_domain"]

In [3]:
X = train[FINAL_COLS]
y = (train["status"] == "phishing").astype(int)
groups = train["group"]
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

rf = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                   RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))

In [4]:
# Probabilité « hors-pli » pour chaque URL du train : prédite par un modèle qui ne l'a jamais vue
proba = cross_val_predict(rf, X, y, groups=groups, cv=cv, method="predict_proba")[:, 1]

print("F1 (seuil 0,5) :", round(f1_score(y, proba >= 0.5), 4))
print("ROC-AUC :", round(roc_auc_score(y, proba), 4))

F1 (seuil 0,5) : 0.9174
ROC-AUC : 0.9729


In [5]:
from sklearn.metrics import confusion_matrix

In [6]:
# Ce que donne le seuil par défaut
tn, fp, fn, tp = confusion_matrix(y, proba >= 0.5).ravel()
print(f"Seuil 0,5 : rappel {tp / (tp + fn):.3f} | taux de faux positifs {fp / (fp + tn):.2%} "
      f"({fp} fausses alertes sur {fp + tn} légitimes)\n")

Seuil 0,5 : rappel 0.916 | taux de faux positifs 7.85% (359 fausses alertes sur 4571 légitimes)



In [7]:
# Le meilleur rappel possible pour un taux de faux positifs maximal donné
fpr, tpr, thresholds = roc_curve(y, proba)
rows = []
for max_fpr in [0.001, 0.01, 0.05]:
    i = np.searchsorted(fpr, max_fpr, side="right") - 1
    rows.append({"FP max": f"{max_fpr:.1%}", "seuil": round(thresholds[i], 3),
                 "rappel": round(tpr[i], 3), "FP réel": f"{fpr[i]:.2%}"})
pd.DataFrame(rows)

,FP max,seuil,rappel,FP réel
0,0.1%,0.933,0.312,0.09%
1,1.0%,0.810,0.611,0.92%
2,5.0%,0.580,0.881,4.97%


In [8]:
def precision_at_prevalence(recall, fpr, prevalence=0.01):
    true_alerts = recall * prevalence            # phishing détecté
    false_alerts = fpr * (1 - prevalence)        # légitimes signalés à tort
    return true_alerts / (true_alerts + false_alerts)

In [9]:
for thr in [0.5, 0.580, 0.810, 0.933]:
    pred = proba >= thr
    rec = pred[y == 1].mean()
    fpr_ = pred[y == 0].mean()
    print(f"seuil {thr:.3f} | rappel {rec:.3f} | FPR {fpr_:.2%} | "
          f"précision si 50 % de phishing : {precision_at_prevalence(rec, fpr_, 0.5):.2f} | "
          f"si 1 % : {precision_at_prevalence(rec, fpr_, 0.01):.2f}")

seuil 0.500 | rappel 0.916 | FPR 7.85% | précision si 50 % de phishing : 0.92 | si 1 % : 0.11
seuil 0.580 | rappel 0.881 | FPR 4.97% | précision si 50 % de phishing : 0.95 | si 1 % : 0.15
seuil 0.810 | rappel 0.611 | FPR 0.92% | précision si 50 % de phishing : 0.99 | si 1 % : 0.40
seuil 0.933 | rappel 0.312 | FPR 0.09% | précision si 50 % de phishing : 1.00 | si 1 % : 0.78
